#### **DPR**

In [1]:
%load_ext autoreload
%autoreload 2

# from src.retrievers.models.encoder import TransformerEncoder
from src.retrievers.dpr import DPR

from hydra import initialize, compose

with initialize(config_path="config", version_base=None):
    cfg = compose(config_name="config")

dpr = DPR(cfg)

/Users/rasmuslinnemann/Desktop/Courses/Semester7/02456 Deep Learning/Project/02456-retrieval-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1123 14:45:49.042000 75361 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6055.94it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.

Loading the dataset

In [2]:
from src.data_utils.corpus import Corpus

corpus = Corpus(dataset_name=cfg.dataset.name, base_dir=f'{cfg.base_path}/retrieval_data')

In [3]:
corpus.ds

DatasetDict({
    train: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 5500
    })
    validation: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 500
    })
    test: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 648
    })
})

Loading the queries for all 3 splits, as in `main.py`

In [4]:
from src.data_utils.query import Query

### Prepare queries
train_queries = [Query(corpus.train['question'][idx], 
                        id=corpus.train['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.train['query_id'][idx]]
                        ) 
                    for idx in range(corpus.train.num_rows)
                ]

val_queries =   [Query(corpus.val['question'][idx], 
                        id=corpus.val['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.val['query_id'][idx]]
                        ) 
                    for idx in range(corpus.val.num_rows)
                ]

test_queries =  [Query(corpus.test['question'][idx], 
                        id=corpus.test['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.test['query_id'][idx]]
                        ) 
                    for idx in range(corpus.test.num_rows)
                ]

From each query, we can get the ground-truth documents using `.relevant_document_ids`.

In [5]:
print(train_queries[5].text)
print(train_queries[5].relevant_document_ids)

Hobby vs. Business
['doc7', 'doc8']


Now what we are missing is to vectorize both the query and all the documents in the corpus, so that we can compare the query vector to all the document vectors and retrieve the most similar documents.

* Since we are dealing with a small dataset, we don't need a full vector database, but we can use a standard numpy array.

In [6]:
# query    → query BERT → pool → q  (768)
# document → doc BERT   → pool → d  (768)
# score = q · d          ← dot product; high means relevant


We begin by embedding the query

* First step is to tokenize the sentence (the string is converted to numbers (e.g. tokens))

In [ ]:
q_tokens = dpr.query_encoder.tokenizer(train_queries[5].text, return_tensors="pt").to(dpr.device)
print(q_tokens["input_ids"])
print(q_tokens["input_ids"].shape)

tensor([[  101, 17792,  5443,  1012,  2449,   102]], device='mps:0')
torch.Size([1, 6])


* Second step is to run the tokenized tensor through BERT which computes attention for each token against every other token and creates a vector embedding for each token

In [10]:
q_embedding = dpr.query_encoder.model(**q_tokens)
print(q_embedding.last_hidden_state.shape)

torch.Size([1, 6, 768])


* Since we only want one vector representing the query, we perform pooling.

In [11]:
q_pooled = q_embedding.pooler_output
print(q_pooled.shape)

torch.Size([1, 768])


That is how we encode a query manually, but we have a function for it `encode_queries` which takes multiple queries as a batch.

If we want to embed the documents, we do more or less the exact same thing.

* Just for the sake of the example, we can embed the ground truth documents of q5, so we can expect a high similarity to the query

In [12]:
print(train_queries[5].id)
print(train_queries[5].relevant_document_ids)

q5
['doc7', 'doc8']


In [15]:
print(corpus.train[5]["text"][0]) #doc7
print(corpus.train[5]["text"][1]) #doc8

Miscellaneous income -- same category used for hobbies.
"You can list it as other income reported on line 21 of form 1040.  In TurboTax, enter at: - Federal Taxes tab (Personal in  Home & Business) - Wages & Income -“I’ll choose what I work on” Button Scroll down to: -Less Common Income -Misc Income, 1099-A, 1099-C. -The next screen will give you several choices. Choose ""Other reportable Income"". You will reach a screen where you can type a description of the income and the amount. Type in the amount of income and categorize as Tutoring."


Embedding doc7. This can also be done using `encode_documents`, if we want to embed multiple documents at once.

In [16]:
d_tokens = dpr.doc_encoder.tokenizer(corpus.train[5]["text"][0], return_tensors="pt").to(dpr.device)
print(d_tokens["input_ids"].shape)

d_embedding = dpr.doc_encoder.model(**d_tokens)
print(d_embedding.last_hidden_state.shape)

d_pooled = d_embedding.pooler_output
print(d_pooled.shape)

torch.Size([1, 13])
torch.Size([1, 13, 768])
torch.Size([1, 768])


Now that we have both a query and a document, we can compute the similarity

In [17]:
q_pooled @ d_pooled.T

tensor([[301.0612]], device='mps:0', grad_fn=<MmBackward0>)

This similarity is what is computed in `query_document` too.

Now we are ready to start training.

In [ ]:
from src.training.loss import DPRLoss
from src.training.batching import build_batch

criterion = DPRLoss()
q_texts, d_texts, pos_mask = build_batch(corpus.train[0:10])

print(len(d_texts))                 # all positives of q0..q9
print(pos_mask[5].nonzero())        # columns holding doc7 and doc8

q = dpr.encode_queries(q_texts)
d = dpr.encode_documents(d_texts)
loss = criterion(q, d, pos_mask)
loss

15
tensor([[7],
        [8]])


tensor(60.7790, device='mps:0', grad_fn=<MeanBackward0>)

In [19]:
import torch

rows = corpus.train[0:10]
q_texts = rows['question']
# print(q_texts)

col = {}                                  # document_id -> column index
d_texts = []
for dids, texts in zip(rows['document_id'], rows['text']):
    # print('------------------')
    for did, txt in zip(dids, texts):
        if did not in col:
            col[did] = len(d_texts)
            print(col)
            d_texts.append(txt)
            print(d_texts)

pos_mask = torch.zeros(len(q_texts), len(d_texts), dtype=torch.bool)
for i, dids in enumerate(rows['document_id']):
    for did in dids:
        pos_mask[i, col[did]] = True

q_texts, d_texts, pos_mask

print(pos_mask.shape)

{'doc0': 0}
['The IRS Guidance pertaining to the subject.  In general the best I can say is your business expense may be deductible.  But it depends on the circumstances and what it is you want to deduct. Travel Taxpayers who travel away from home on business may deduct related   expenses, including the cost of reaching their destination, the cost   of lodging and meals and other ordinary and necessary expenses.   Taxpayers are considered “traveling away from home” if their duties   require them to be away from home substantially longer than an   ordinary day’s work and they need to sleep or rest to meet the demands   of their work. The actual cost of meals and incidental expenses may be   deducted or the taxpayer may use a standard meal allowance and reduced   record keeping requirements. Regardless of the method used, meal   deductions are generally limited to 50 percent as stated earlier.    Only actual costs for lodging may be claimed as an expense and   receipts must be kept for d

In [20]:
DatasetDict({
    train: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 5500
    })
    validation: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 500
    })
    test: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 648
    })
})

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1010563606.py, line 2)

In [24]:
print(corpus.train[5])

{'question': 'Hobby vs. Business', 'text': ['Miscellaneous income -- same category used for hobbies.', '"You can list it as other income reported on line 21 of form 1040.  In TurboTax, enter at: - Federal Taxes tab (Personal in  Home & Business) - Wages & Income -“I’ll choose what I work on” Button Scroll down to: -Less Common Income -Misc Income, 1099-A, 1099-C. -The next screen will give you several choices. Choose ""Other reportable Income"". You will reach a screen where you can type a description of the income and the amount. Type in the amount of income and categorize as Tutoring."'], 'query_id': 'q5', 'document_id': ['doc7', 'doc8']}


In [38]:
rows = corpus.train[10:20]

In [ ]:
def build_batch(rows):
    queries_text = rows['question']
    unique_documents = []
    unique_documents_text = []

    for i, doc_list in enumerate(rows['document_id']):
        for j, doc_id in enumerate(doc_list):
            if doc_id not in unique_documents:
                unique_documents.append(doc_id)
                unique_documents_text.append(rows['text'][i][j])

    mask = torch.zeros(
        (len(rows['query_id']), len(unique_documents)),
        dtype=torch.bool)

    document_to_idx = {doc_id: idx for idx, doc_id in enumerate(unique_documents)}

    for i, doc_list in enumerate(rows['document_id']):
        for doc_id in doc_list:
            mask[i, document_to_idx[doc_id]] = True

    return queries_text, unique_documents_text,  mask